# Complete Occulus Workflow — End to End

This capstone notebook combines every major Occulus module into one coherent
pipeline.  It works with both synthetic data (default) and real LiDAR via AbovePy
or USGS 3DEP.

**Pipeline**:
```
Input LiDAR (LAS/LAZ/XYZ)
    ↓ occulus.io.read
    ↓ occulus.filters (SOR + voxel downsample)
    ↓ occulus.segmentation.classify_ground_csf  ← CSF ground
    ↓ occulus.normals.estimate_normals
    ↓ occulus.metrics.canopy_height_model        ← CHM
    ↓ occulus.segmentation.segment_trees         ← individual trees
    ↓ occulus.features.compute_geometric_features ← eigenvalue features
    ↓ occulus.metrics.coverage_statistics        ← gap fraction, density
    ↓ occulus.mesh.poisson_mesh (optional)       ← terrain mesh
    ↓ occulus.io.write                           ← export
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path

OUTPUT_DIR = Path('../../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

timings = {}

## Data Source Selection

Choose one of: AbovePy (KY From Above), USGS 3DEP, or synthetic data.

In [ ]:
SOURCE = 'synthetic'  # 'abovepy' | 'usgs' | 'synthetic'

if SOURCE == 'abovepy':
    from abovepy import search, download
    import tempfile
    tiles = search('lidar', county='bell', phase=3)  # Bell County, SE KY (Cumberland Gap)
    path = download(tiles[0], dest=Path(tempfile.mkdtemp()))
    from occulus.io import read
    cloud_raw = read(path, platform='aerial', subsample=0.2)

elif SOURCE == 'usgs':
    import urllib.request, tempfile
    url = ('https://rockyweb.usgs.gov/vdelivery/Datasets/Staged/Elevation/LPC/Projects/'
           'USGS_LPC_KY_Statewide_2019/laz/USGS_LPC_KY_Statewide_2019_e1380n4170.laz')
    dest = Path(tempfile.mkdtemp()) / Path(url).name
    urllib.request.urlretrieve(url, str(dest))
    from occulus.io import read
    cloud_raw = read(dest, platform='aerial', subsample=0.25)

else:  # synthetic
    from occulus.types import AerialCloud
    rng = np.random.default_rng(0)
    n = 50_000
    x = rng.uniform(0, 150, n)
    y = rng.uniform(0, 150, n)
    z = 3 * np.sin(x / 40) + 2 * np.cos(y / 35) + rng.normal(0, 0.05, n)
    for _ in range(40):
        tx, ty, th = rng.uniform(5, 145), rng.uniform(5, 145), rng.uniform(5, 30)
        mask = np.hypot(x - tx, y - ty) < 2.5
        z[mask] += rng.uniform(0, th, mask.sum())
    cloud_raw = AerialCloud(np.column_stack([x, y, np.maximum(z, 0)]).astype(np.float64))

print(f'[Source: {SOURCE}]  Loaded: {cloud_raw}')

## Step 1: Filter

In [ ]:
from occulus.filters import statistical_outlier_removal, voxel_downsample

t0 = time.perf_counter()
clean = statistical_outlier_removal(cloud_raw, nb_neighbors=16, std_ratio=2.5)
ds = voxel_downsample(clean, voxel_size=0.3)
timings['filter'] = time.perf_counter() - t0
print(f'Raw→clean→downsampled: {cloud_raw.n_points:,} → {clean.n_points:,} → {ds.n_points:,}')

## Step 2: Ground Classification

In [ ]:
from occulus.segmentation import classify_ground_csf
from occulus.types import AerialCloud

t0 = time.perf_counter()
classified = classify_ground_csf(ds)
timings['ground'] = time.perf_counter() - t0

if isinstance(classified, AerialCloud) and classified.classification is not None:
    n_g = int((classified.classification == 2).sum())
    print(f'Ground: {n_g:,} ({n_g/ds.n_points:.1%})  |  Above: {ds.n_points-n_g:,}')

## Step 3: Normals + CHM + Trees

In [ ]:
from occulus.normals import estimate_normals
from occulus.metrics import canopy_height_model, coverage_statistics, compute_cloud_statistics
from occulus.segmentation import segment_trees

t0 = time.perf_counter()
ds_n = estimate_normals(ds, radius=1.5)
chm, xe, ye = canopy_height_model(classified, resolution=0.5)
cov = coverage_statistics(classified, resolution=0.5)
seg = segment_trees(classified, resolution=0.5, min_height=3.0)
timings['analysis'] = time.perf_counter() - t0

valid_chm = chm[~np.isnan(chm)]
print(f'CHM: {chm.shape[0]}×{chm.shape[1]}')
print(f'Max canopy height : {valid_chm.max():.1f} m')
print(f'Gap fraction      : {cov.gap_fraction:.2%}')
print(f'Trees detected    : {seg.n_segments}')

## Step 4: Geometric Features

In [ ]:
from occulus.features import compute_geometric_features

t0 = time.perf_counter()
feats = compute_geometric_features(ds_n, radius=1.5)
timings['features'] = time.perf_counter() - t0

print(f'Features: planarity {feats.planarity.mean():.3f}, sphericity {feats.sphericity.mean():.3f}, curvature {feats.curvature.mean():.4f}')

## Results Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 4, hspace=0.35, wspace=0.35)

ax_chm = fig.add_subplot(gs[0, :2])
ax_seg = fig.add_subplot(gs[0, 2:])
ax_plan = fig.add_subplot(gs[1, 0])
ax_sph  = fig.add_subplot(gs[1, 1])
ax_curv = fig.add_subplot(gs[1, 2])
ax_bar  = fig.add_subplot(gs[1, 3])

# CHM
im0 = ax_chm.imshow(chm, origin='lower', cmap='Greens',
                    extent=[xe[0], xe[-1], ye[0], ye[-1]])
plt.colorbar(im0, ax=ax_chm, label='m'); ax_chm.set_title('Canopy Height Model')

# Segmentation
lbl = seg.labels
xyz_ds = ds.xyz
above = lbl != -1
ax_seg.scatter(xyz_ds[~above, 0], xyz_ds[~above, 1], s=0.3, c='tan', alpha=0.3)
if above.any():
    ax_seg.scatter(xyz_ds[above, 0], xyz_ds[above, 1], c=lbl[above]%20, cmap='tab20', s=0.5, alpha=0.7)
ax_seg.set_title(f'Tree Segmentation ({seg.n_segments} trees)')

# Feature maps
for ax, arr, name, cmap in [
    (ax_plan, feats.planarity, 'Planarity', 'Blues'),
    (ax_sph, feats.sphericity, 'Sphericity', 'Greens'),
    (ax_curv, feats.curvature, 'Curvature', 'plasma'),
]:
    sc = ax.scatter(xyz_ds[:, 0], xyz_ds[:, 1], c=arr, cmap=cmap, s=0.5)
    plt.colorbar(sc, ax=ax, shrink=0.8); ax.set_title(name)

# Timing bar chart
steps = list(timings.keys())
vals = list(timings.values())
ax_bar.barh(steps, vals, color='steelblue')
ax_bar.set_xlabel('Seconds'); ax_bar.set_title('Processing Time')
for i, v in enumerate(vals):
    ax_bar.text(v + 0.01, i, f'{v:.2f}s', va='center', fontsize=8)

fig.suptitle('Occulus — Complete Workflow Dashboard', fontsize=15, fontweight='bold')
plt.savefig(OUTPUT_DIR / 'complete_workflow.png', dpi=150, bbox_inches='tight')
plt.show()

## Final Summary

In [ ]:
stats = compute_cloud_statistics(cloud_raw)
total_time = sum(timings.values())

print('╔══════════════════════════════════════════════╗')
print('║    OCCULUS — COMPLETE WORKFLOW SUMMARY      ║')
print('╚══════════════════════════════════════════════╝')
print(f'  Input points     : {cloud_raw.n_points:,}')
print(f'  After filter     : {ds.n_points:,}')
print(f'  Z range          : {stats.z_min:.1f} – {stats.z_max:.1f} m')
if isinstance(classified, AerialCloud) and classified.classification is not None:
    n_g = int((classified.classification == 2).sum())
    print(f'  Ground points    : {n_g:,} ({n_g/ds.n_points:.1%})')
print(f'  Trees detected   : {seg.n_segments}')
print(f'  Max canopy ht    : {valid_chm.max():.1f} m')
print(f'  Gap fraction     : {cov.gap_fraction:.2%}')
print(f'  Mean planarity   : {feats.planarity.mean():.3f}')
print(f'  Total time       : {total_time:.2f} s')
print(f'  Output image     : {OUTPUT_DIR}/complete_workflow.png')